# NB6 — Human Annotation (Tagged Recovery & Scoring)

**Purpose:** Recover the original 100-example annotation batch from the completed Annotator A CSV, Annotator B Excel sheet, and original automatic-label `key.json`; calculate inter-annotator and human–automatic agreement; preserve a reproducible, version-tagged record.

## Tag-based workflow
- `SAMPLE_MODE = "recover_existing"` scores the already completed original batch. It does not draw a new sample.
- `SAMPLE_MODE = "generate_new"` deliberately creates a new sample from the authoritative NB4 v2 dataset, with a distinct `SAMPLE_TAG`.
- `REGENERATE_SAMPLE` is an explicit guard. New sampling is blocked unless it is `True`.
- Every run has a `RUN_TAG`; outputs and Drive snapshots are versioned and never silently overwrite prior runs.
- Completed annotation uploads are retained as incoming originals.

**Important provenance note:** The project owner confirms that Annotator B completed the annotation fairly and independently despite the exact match with the automatic labels. This is recorded as owner-provided context. Agreement statistics alone cannot establish whether annotation was independent.

## 1. Configuration — choose the tagged mode

In [36]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, shutil, math, random
import numpy as np
import pandas as pd

# ----- USER-CONTROLLED TAGS -----
SAMPLE_MODE = "recover_existing"  # "recover_existing" or "generate_new"
SAMPLE_TAG = "original_100_annotator_batch"  # change for every intentionally new sample
RUN_TAG = "recovery_scoring_v1"                # change for a distinct scoring/export run
REGENERATE_SAMPLE = False                      # must be True to generate a new sample
SAMPLE_SIZE = 100
SAMPLE_SEED = 0

# Record this provenance statement because the project owner explicitly confirmed it.
ANNOTATOR_B_INDEPENDENCE_NOTE = (
    "Project owner confirms Annotator B completed the annotations fairly and independently; "
    "the exact match with automatic labels is not being treated as evidence of misconduct."
)

PROJECT_ROOT = Path("/content/irrev")
NB6_ROOT = PROJECT_ROOT / "data" / "human_annotation"
INCOMING = NB6_ROOT / "incoming_uploads" / SAMPLE_TAG
SAMPLE_DIR = NB6_ROOT / "samples" / SAMPLE_TAG
RESULTS = PROJECT_ROOT / "results" / "human_annotation" / RUN_TAG
REPORTS = PROJECT_ROOT / "reports" / "nb06_human_annotation" / RUN_TAG
for p in [INCOMING, SAMPLE_DIR, RESULTS, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print("SAMPLE_MODE:", SAMPLE_MODE)
print("SAMPLE_TAG:", SAMPLE_TAG)
print("RUN_TAG:", RUN_TAG)
print("REGENERATE_SAMPLE:", REGENERATE_SAMPLE)

SAMPLE_MODE: recover_existing
SAMPLE_TAG: original_100_annotator_batch
RUN_TAG: recovery_scoring_v1
REGENERATE_SAMPLE: False


## 2. Mount Drive and establish source locations

In [37]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_AVAILABLE = True
except Exception as e:
    DRIVE_AVAILABLE = False
    print("Drive mount unavailable:", repr(e))

DRIVE_ROOT = Path("/content/drive/MyDrive/ir")
BACKUP_ROOT = Path("/content/drive/MyDrive/ir_final")
NB4_SNAPSHOT_NAME = "NB4_v2_snapshot_20261118T194353Z"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Drive available:", DRIVE_AVAILABLE)
print("Main project Drive:", DRIVE_ROOT)
print("Independent backup Drive:", BACKUP_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive available: True
Main project Drive: /content/drive/MyDrive/ir
Independent backup Drive: /content/drive/MyDrive/ir_final


## 3. Upload original annotation files and key (recovery mode)

In [38]:
if SAMPLE_MODE == "recover_existing":
    from google.colab import files
    from pathlib import Path
    import shutil

    # ---------------------------------------------------------
    # UPLOAD 1 — ANNOTATOR A
    # ---------------------------------------------------------
    print("=" * 65)
    print("UPLOAD 1/3 — ANNOTATOR A COMPLETED CSV")
    print("=" * 65)

    uploaded_a = files.upload()

    csvs = [n for n in uploaded_a if n.lower().endswith(".csv")]

    if len(uploaded_a) != 1 or len(csvs) != 1:
        raise ValueError("Upload exactly one CSV file for Annotator A.")

    A_UPLOAD = csvs[0]

    # ---------------------------------------------------------
    # UPLOAD 2 — ANNOTATOR B
    # ---------------------------------------------------------
    print("\n" + "=" * 65)
    print("UPLOAD 2/3 — ANNOTATOR B COMPLETED EXCEL")
    print("=" * 65)

    uploaded_b = files.upload()

    excels = [
        n for n in uploaded_b
        if n.lower().endswith((".xlsx", ".xls"))
    ]

    if len(uploaded_b) != 1 or len(excels) != 1:
        raise ValueError("Upload exactly one Excel file for Annotator B.")

    B_UPLOAD = excels[0]

    # ---------------------------------------------------------
    # UPLOAD 3 — AUTOMATIC-LABEL KEY JSON
    # ---------------------------------------------------------
    print("\n" + "=" * 65)
    print("UPLOAD 3/3 — ORIGINAL AUTOMATIC-LABEL KEY JSON")
    print("=" * 65)

    uploaded_key = files.upload()

    jsons = [
        n for n in uploaded_key
        if n.lower().endswith(".json")
    ]

    if len(uploaded_key) != 1 or len(jsons) != 1:
        raise ValueError("Upload exactly one automatic-label key JSON file.")

    KEY_UPLOAD = jsons[0]

    # ---------------------------------------------------------
    # PRESERVE ORIGINAL UPLOADS
    # ---------------------------------------------------------
    A_ORIGINAL = INCOMING / "annotator_A_completed_original.csv"
    B_ORIGINAL = INCOMING / "annotator_B_completed_original.xlsx"
    KEY_ORIGINAL = INCOMING / "automatic_label_key_original.json"

    shutil.copy2(A_UPLOAD, A_ORIGINAL)
    shutil.copy2(B_UPLOAD, B_ORIGINAL)
    shutil.copy2(KEY_UPLOAD, KEY_ORIGINAL)

    # ---------------------------------------------------------
    # VERIFY PRESERVED FILES
    # ---------------------------------------------------------
    print("\n" + "=" * 65)
    print("ORIGINAL UPLOAD PRESERVATION")
    print("=" * 65)

    preserved_files = [
        A_ORIGINAL,
        B_ORIGINAL,
        KEY_ORIGINAL
    ]

    for p in preserved_files:
        if not p.exists():
            raise FileNotFoundError(
                f"Original upload was not preserved: {p}"
            )

        print(f"\nFile: {p.name}")
        print(f"Path: {p}")
        print(f"Size: {p.stat().st_size:,} bytes")
        print(f"SHA-256: {sha256_file(p)}")

    print("\nAll three original uploads preserved successfully.")

else:
    print(
        "Recovery uploads skipped: "
        "SAMPLE_MODE is generate_new."
    )

UPLOAD 1/3 — ANNOTATOR A COMPLETED CSV


Saving sheet_A.csv to sheet_A (4).csv

UPLOAD 2/3 — ANNOTATOR B COMPLETED EXCEL


Saving sheet_B.xlsx to sheet_B (4).xlsx

UPLOAD 3/3 — ORIGINAL AUTOMATIC-LABEL KEY JSON


Saving key.json to key (1).json

ORIGINAL UPLOAD PRESERVATION

File: annotator_A_completed_original.csv
Path: /content/irrev/data/human_annotation/incoming_uploads/original_100_annotator_batch/annotator_A_completed_original.csv
Size: 58,245 bytes
SHA-256: 079c2e29839f055c4153fa1e54630381ac0715cbebf0b5708ecaa6dde2df76b8

File: annotator_B_completed_original.xlsx
Path: /content/irrev/data/human_annotation/incoming_uploads/original_100_annotator_batch/annotator_B_completed_original.xlsx
Size: 20,946 bytes
SHA-256: 241c4dae2b16333ff919a6829b304afaf3fcda9f10051970051cea358eba5d6e

File: automatic_label_key_original.json
Path: /content/irrev/data/human_annotation/incoming_uploads/original_100_annotator_batch/automatic_label_key_original.json
Size: 3,042 bytes
SHA-256: be4e5d0117e703c97208c47cd68ce1c2d0526db0b9218f876bb34c3563e74578

All three original uploads preserved successfully.


## 4. Load, validate, and freeze the original batch (recovery mode)

In [39]:
key_suffix = Path(KEY_ORIGINAL).suffix.lower()

if key_suffix == ".csv":
    key_df = pd.read_csv(KEY_ORIGINAL, dtype=str)
elif key_suffix in [".xlsx", ".xls"]:
    key_df = pd.read_excel(KEY_ORIGINAL, dtype=str)
elif key_suffix == ".json":
    with open(KEY_ORIGINAL, "r", encoding="utf-8") as f:
        key_data = json.load(f)

    if isinstance(key_data, dict):
        key_df = pd.DataFrame(
            list(key_data.items()),
            columns=["id", "automatic_label"]
        )
    elif isinstance(key_data, list):
        key_df = pd.DataFrame(key_data)
    else:
        raise ValueError("Unsupported JSON key-sheet structure.")
else:
    raise ValueError(f"Unsupported key-sheet format: {key_suffix}")

# Normalize source column names
key_df.columns = [
    str(c).strip().lower().replace(" ", "_").replace("-", "_")
    for c in key_df.columns
]

print("Original key-sheet columns:", list(key_df.columns))

# Identify ID and automatic-label columns
id_candidates = ["id", "record_id", "sample_id", "example_id"]
label_candidates = [
    "automatic_label", "auto_label", "automatic",
    "label", "ground_truth"
]

id_cols = [c for c in id_candidates if c in key_df.columns]
label_cols = [c for c in label_candidates if c in key_df.columns]

if len(id_cols) != 1 or len(label_cols) != 1:
    raise ValueError(
        f"Cannot uniquely identify key columns. "
        f"Available columns: {list(key_df.columns)}"
    )

key_id_col = id_cols[0]
key_label_col = label_cols[0]

# Create a clean working copy
key_df = key_df[[key_id_col, key_label_col]].copy()
key_df.columns = ["id", "automatic_label"]

key_df["id"] = key_df["id"].astype("string").str.strip()
key_df["automatic_label"] = (
    key_df["automatic_label"].astype("string").str.strip().str.lower()
)

# Inspect missing values
missing_key_rows = key_df[
    key_df["id"].isna()
    | (key_df["id"] == "")
    | key_df["automatic_label"].isna()
    | (key_df["automatic_label"] == "")
]

if not missing_key_rows.empty:
    print("Rows requiring inspection:")
    display(missing_key_rows)
    raise ValueError("Blank IDs or labels found in original key sheet.")

# Validate labels
invalid_key = sorted(
    set(key_df["automatic_label"].tolist()) - ALLOWED_LABELS
)

if invalid_key:
    raise ValueError(f"Invalid automatic labels: {invalid_key}")

# Validate uniqueness and sample size
if key_df["id"].duplicated().any():
    raise ValueError("Duplicate IDs found in automatic-label key.")

if len(key_df) != SAMPLE_SIZE:
    raise ValueError(
        f"Key has {len(key_df)} rows; expected {SAMPLE_SIZE}."
    )

automatic_key = dict(
    zip(key_df["id"], key_df["automatic_label"])
)

print("Automatic-label key validated.")
print("Records:", len(automatic_key))
print("Label distribution:")
print(pd.Series(list(automatic_key.values())).value_counts())

Original key-sheet columns: ['id', 'automatic_label']
Automatic-label key validated.
Records: 100
Label distribution:
reversible              51
irreversible            31
partially_reversible    18
Name: count, dtype: int64


In [40]:
from pathlib import Path
import json
import pandas as pd

# Load the JSON key correctly
with open(KEY_ORIGINAL, "r", encoding="utf-8") as f:
    automatic_key = json.load(f)

if not isinstance(automatic_key, dict):
    raise TypeError("Expected the automatic-label JSON to contain an ID-to-label dictionary.")

key_df_raw = pd.DataFrame(
    list(automatic_key.items()),
    columns=["id", "label"]
)

# Clean whitespace
key_df_raw["id"] = key_df_raw["id"].astype(str).str.strip()
key_df_raw["label"] = key_df_raw["label"].astype(str).str.strip()

print("=" * 65)
print("AUTOMATIC-LABEL KEY VALIDATION")
print("=" * 65)

print("\nFILE:", KEY_ORIGINAL)
print("SHAPE:", key_df_raw.shape)
print("COLUMNS:", list(key_df_raw.columns))

print("\nNON-EMPTY VALUE COUNTS:")
for col in key_df_raw.columns:
    count = key_df_raw[col].ne("").sum()
    print(f"{col}: {count}/{len(key_df_raw)}")

print("\nLABEL DISTRIBUTION:")
print(key_df_raw["label"].value_counts())

print("\nFIRST 10 ROWS:")
display(key_df_raw.head(10))

print("\nLAST 10 ROWS:")
display(key_df_raw.tail(10))

# Integrity checks
allowed_labels = {
    "reversible",
    "partially_reversible",
    "irreversible"
}

assert len(key_df_raw) == 100, f"Expected 100 labels, got {len(key_df_raw)}"
assert key_df_raw["id"].is_unique, "Duplicate IDs found."
assert key_df_raw["id"].ne("").all(), "Blank IDs found."
assert key_df_raw["label"].ne("").all(), "Blank labels found."
assert set(key_df_raw["label"]).issubset(allowed_labels), "Unexpected labels found."

print("\nSUCCESS: 100 valid automatic labels loaded.")

AUTOMATIC-LABEL KEY VALIDATION

FILE: /content/irrev/data/human_annotation/incoming_uploads/original_100_annotator_batch/automatic_label_key_original.json
SHAPE: (100, 2)
COLUMNS: ['id', 'label']

NON-EMPTY VALUE COUNTS:
id: 100/100
label: 100/100

LABEL DISTRIBUTION:
label
reversible              51
irreversible            31
partially_reversible    18
Name: count, dtype: int64

FIRST 10 ROWS:


,id,label
0,sbx00837_0,partially_reversible
1,sbx00068_1,reversible
2,sbx01329_0,irreversible
3,sbx00459_0,irreversible
4,sbx00977_0,irreversible
5,sbx00506_0,irreversible
6,sbx01261_1,partially_reversible
7,sbx00548_1,irreversible
8,sbx00270_0,reversible
9,sbx00128_0,reversible



LAST 10 ROWS:


,id,label
90,sbx00387_0,irreversible
91,sbx01215_0,reversible
92,sbx01006_1,reversible
93,sbx01219_0,irreversible
94,sbx00270_1,irreversible
95,sbx00444_0,partially_reversible
96,sbx01123_1,partially_reversible
97,sbx01369_0,reversible
98,sbx00061_1,irreversible
99,sbx00933_1,reversible



SUCCESS: 100 valid automatic labels loaded.


## 5. Optional deliberate new sample generation (tagged; never implicit)

In [41]:
if SAMPLE_MODE == "generate_new":
    if not REGENERATE_SAMPLE:
        raise RuntimeError(
            "New sampling is disabled. Set REGENERATE_SAMPLE=True and change SAMPLE_TAG "
            "to a new unique tag only when you intentionally want a different annotation batch."
        )
    if SAMPLE_TAG == "original_100_annotator_batch":
        raise RuntimeError("Do not reuse the original sample tag for a new sample.")

    candidates = [
        DRIVE_ROOT / NB4_SNAPSHOT_NAME / "integrated_dataset_v2.jsonl",
        BACKUP_ROOT / NB4_SNAPSHOT_NAME / "integrated_dataset_v2.jsonl",
    ]
    source_path = next((p for p in candidates if p.exists()), None)
    if source_path is None:
        raise FileNotFoundError("Authoritative NB4 v2 integrated dataset not found in either Drive snapshot.")
    source_rows = [json.loads(line) for line in source_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    labelled = [r for r in source_rows if r.get("label") in ALLOWED_LABELS]
    rng = random.Random(SAMPLE_SEED)
    selected = rng.sample(labelled, SAMPLE_SIZE)
    sample_df = pd.DataFrame([{
        "id": str(r["id"]),
        "environment": r.get("environment", ""),
        "state_text": r.get("state_text", ""),
        "action_text": r.get("action_text", ""),
        "automatic_label": r["label"]
    } for r in selected])
    if sample_df.id.duplicated().any():
        raise ValueError("Generated sample contains duplicate IDs.")
    sample_df.to_csv(SAMPLE_DIR / "generated_sample_private.csv", index=False)
    (SAMPLE_DIR / "generated_automatic_key.json").write_text(
        json.dumps(dict(zip(sample_df.id, sample_df.automatic_label)), indent=2), encoding="utf-8"
    )
    (SAMPLE_DIR / "generation_manifest.json").write_text(json.dumps({
        "sample_tag": SAMPLE_TAG, "sample_seed": SAMPLE_SEED, "sample_size": SAMPLE_SIZE,
        "source_snapshot": NB4_SNAPSHOT_NAME, "source_file": str(source_path),
        "source_sha256": sha256_file(source_path), "generated_at_utc": datetime.now(timezone.utc).isoformat()
    }, indent=2), encoding="utf-8")
    print("New tagged sample created:", SAMPLE_TAG)
    print("Private sample and automatic key saved. Create blinded annotator copies in a separate controlled handoff.")
else:
    print("No new sample generated. Original annotation batch remains selected.")

No new sample generated. Original annotation batch remains selected.


## 6. Agreement metrics — three-class and binary

In [42]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix,
    accuracy_score
)

LABEL_ORDER = [
    "reversible",
    "partially_reversible",
    "irreversible"
]

BINARY_ORDER = ["reversible", "not_reversible"]

# =============================================================
# LOAD AND VALIDATE ALL THREE SOURCES
# =============================================================

if SAMPLE_MODE == "recover_existing":

    # Load annotator sheets
    ann_a = pd.read_csv(
        A_ORIGINAL, dtype=str, keep_default_na=False
    )

    ann_b = pd.read_excel(
        B_ORIGINAL, dtype=str
    ).fillna("")

    # Load automatic-label JSON dictionary
    with open(KEY_ORIGINAL, "r", encoding="utf-8") as f:
        automatic_key = json.load(f)

    if not isinstance(automatic_key, dict):
        raise TypeError("Automatic-label key must be a JSON dictionary.")

    df_key = pd.DataFrame(
        list(automatic_key.items()),
        columns=["id", "label"]
    )

    # Normalize values
    for df in [ann_a, ann_b, df_key]:
        df["id"] = df["id"].astype(str).str.strip()
        df["label"] = df["label"].astype(str).str.strip()

    # Validate required columns
    for name, df in [
        ("Annotator A", ann_a),
        ("Annotator B", ann_b),
        ("Automatic key", df_key)
    ]:
        if not {"id", "label"}.issubset(df.columns):
            raise ValueError(f"{name} is missing id or label columns.")

        if df["id"].duplicated().any():
            raise ValueError(f"Duplicate IDs in {name}.")

        if df["id"].eq("").any() or df["label"].eq("").any():
            raise ValueError(f"Blank IDs or labels in {name}.")

        invalid = set(df["label"]) - set(LABEL_ORDER)
        if invalid:
            raise ValueError(f"Invalid labels in {name}: {invalid}")

    # Verify identical sample IDs
    ids_a = set(ann_a["id"])
    ids_b = set(ann_b["id"])
    ids_key = set(df_key["id"])

    if not (ids_a == ids_b == ids_key):
        raise ValueError("The three files do not contain identical sample IDs.")

    # Align all labels to Annotator A's ID order
    b_lookup = ann_b.set_index("id")["label"]
    auto_lookup = df_key.set_index("id")["label"]

    A = ann_a["label"].tolist()
    B = ann_a["id"].map(b_lookup).tolist()
    AUTO = ann_a["id"].map(auto_lookup).tolist()

    if any(pd.isna(x) for x in B + AUTO):
        raise ValueError("Missing labels after ID alignment.")

    print("All three sources validated.")
    print("Aligned records:", len(A))

    # =========================================================
    # BINARY LABEL CONVERSION
    # =========================================================

    def to_binary(labels):
        return [
            "reversible" if x == "reversible" else "not_reversible"
            for x in labels
        ]

    # =========================================================
    # BOOTSTRAP KAPPA
    # =========================================================

    def bootstrap_kappa(
        x, y, n_boot=5000, seed=20261119, labels=None
    ):
        x = np.asarray(x, dtype=object)
        y = np.asarray(y, dtype=object)

        rng = np.random.default_rng(seed)
        vals = []
        n = len(x)

        for _ in range(n_boot):
            idx = rng.integers(0, n, n)

            k = cohen_kappa_score(
                x[idx], y[idx], labels=labels
            )

            if np.isfinite(k):
                vals.append(float(k))

        if not vals:
            return [float("nan"), float("nan")]

        return [
            float(np.percentile(vals, 2.5)),
            float(np.percentile(vals, 97.5))
        ]

    # =========================================================
    # COMPARISON FUNCTION
    # =========================================================

    def compare(name, x, y, labels, seed):

        x = np.asarray(x, dtype=object)
        y = np.asarray(y, dtype=object)

        return {
            "comparison": name,
            "n": len(x),
            "exact_agreement": float(accuracy_score(x, y)),
            "cohen_kappa": float(
                cohen_kappa_score(x, y, labels=labels)
            ),
            "bootstrap_95_ci": bootstrap_kappa(
                x, y, labels=labels, seed=seed
            ),
            "confusion_matrix_labels": labels,
            "confusion_matrix": confusion_matrix(
                x, y, labels=labels
            ).tolist()
        }

    # =========================================================
    # SIX AGREEMENT COMPARISONS
    # =========================================================

    comparisons = [
        compare(
            "Annotator A vs Annotator B (3-class)",
            A, B, LABEL_ORDER, 101
        ),
        compare(
            "Annotator A vs automatic (3-class)",
            A, AUTO, LABEL_ORDER, 102
        ),
        compare(
            "Annotator B vs automatic (3-class)",
            B, AUTO, LABEL_ORDER, 103
        ),
        compare(
            "Annotator A vs Annotator B (binary reversible vs not)",
            to_binary(A), to_binary(B), BINARY_ORDER, 104
        ),
        compare(
            "Annotator A vs automatic (binary reversible vs not)",
            to_binary(A), to_binary(AUTO), BINARY_ORDER, 105
        ),
        compare(
            "Annotator B vs automatic (binary reversible vs not)",
            to_binary(B), to_binary(AUTO), BINARY_ORDER, 106
        )
    ]

    # =========================================================
    # RESULTS TABLE
    # =========================================================

    metrics_df = pd.DataFrame([
        {
            "comparison": m["comparison"],
            "n": m["n"],
            "exact_agreement": m["exact_agreement"],
            "cohen_kappa": m["cohen_kappa"],
            "ci_95_lower": m["bootstrap_95_ci"][0],
            "ci_95_upper": m["bootstrap_95_ci"][1]
        }
        for m in comparisons
    ])

    print("\n" + "=" * 75)
    print("NB6 — AGREEMENT ANALYSIS")
    print("=" * 75)

    display(metrics_df)

    # =========================================================
    # SAVE RESULTS
    # =========================================================

    RESULTS.mkdir(parents=True, exist_ok=True)

    (RESULTS / "NB6_agreement_metrics.json").write_text(
        json.dumps(comparisons, indent=2),
        encoding="utf-8"
    )

    metrics_df.to_csv(
        RESULTS / "NB6_agreement_metrics.csv",
        index=False
    )

    print("\nResults saved successfully:")
    print(RESULTS / "NB6_agreement_metrics.json")
    print(RESULTS / "NB6_agreement_metrics.csv")

else:
    print("Agreement analysis skipped: SAMPLE_MODE is generate_new.")

All three sources validated.
Aligned records: 100

NB6 — AGREEMENT ANALYSIS


,comparison,n,exact_agreement,cohen_kappa,ci_95_lower,ci_95_upper
0,Annotator A vs Annotator B (3-class),100,0.80,0.686520,0.565959,0.801407
1,Annotator A vs automatic (3-class),100,0.80,0.686520,0.564267,0.799723
2,Annotator B vs automatic (3-class),100,1.00,1.000000,1.000000,1.000000
3,Annotator A vs Annotator B (binary reversible ...,100,0.88,0.761051,0.632353,0.878690
4,Annotator A vs automatic (binary reversible vs...,100,0.88,0.761051,0.630350,0.879663
5,Annotator B vs automatic (binary reversible vs...,100,1.00,1.000000,1.000000,1.000000



Results saved successfully:
/content/irrev/results/human_annotation/recovery_scoring_v1/NB6_agreement_metrics.json
/content/irrev/results/human_annotation/recovery_scoring_v1/NB6_agreement_metrics.csv


## 7. Detailed counts and disagreement analysis

In [43]:
if SAMPLE_MODE == "recover_existing":
    count_table = pd.DataFrame({
        "Annotator A": ann_a["label"].value_counts().reindex(LABEL_ORDER, fill_value=0),
        "Annotator B": ann_b["label"].value_counts().reindex(LABEL_ORDER, fill_value=0),
        "Automatic": pd.Series(AUTO).value_counts().reindex(LABEL_ORDER, fill_value=0)
    })
    display(count_table)
    disagreement = ann_a[["id", "environment", "state_text", "action_text", "label"]].copy()
    disagreement = disagreement.rename(columns={"label": "annotator_A"})
    disagreement["annotator_B"] = ann_b["label"].values
    disagreement["automatic_label"] = AUTO
    disagreement["A_equals_B"] = disagreement.annotator_A == disagreement.annotator_B
    disagreement["A_equals_auto"] = disagreement.annotator_A == disagreement.automatic_label
    disagreement["B_equals_auto"] = disagreement.annotator_B == disagreement.automatic_label
    disagreement.to_csv(RESULTS / "NB6_row_level_comparison_private.csv", index=False)
    display(disagreement.loc[~disagreement.A_equals_B, ["id", "annotator_A", "annotator_B", "automatic_label"]])
    print("A/B agreements:", int(disagreement.A_equals_B.sum()), "/", len(disagreement))
    print("B/automatic exact matches:", int(disagreement.B_equals_auto.sum()), "/", len(disagreement))

,Annotator A,Annotator B,Automatic
reversible,39,51,51
partially_reversible,20,18,18
irreversible,41,31,31


,id,annotator_A,annotator_B,automatic_label
8,sbx00270_0,irreversible,reversible,reversible
14,sbx01383_0,irreversible,reversible,reversible
17,sbx00947_0,irreversible,reversible,reversible
20,sbx01287_0,partially_reversible,reversible,reversible
22,sbx00500_0,irreversible,reversible,reversible
23,sbx00119_0,partially_reversible,reversible,reversible
24,sbx01179_1,partially_reversible,reversible,reversible
27,sbx00833_0,irreversible,reversible,reversible
32,sbx00247_1,irreversible,partially_reversible,partially_reversible
38,sbx01241_1,irreversible,partially_reversible,partially_reversible


A/B agreements: 80 / 100
B/automatic exact matches: 100 / 100


## 8. Human-readable report

In [44]:
if SAMPLE_MODE == "recover_existing":
    report_lines = [
        "# NB6 — Human Annotation Results",
        "",
        f"- Sample tag: `{SAMPLE_TAG}`",
        f"- Run tag: `{RUN_TAG}`",
        f"- Sample size: {len(ann_a)}",
        "- Sample provenance: recovered from the completed annotator sheets and original automatic-label key supplied for this run.",
        "- Both annotator sheets contain the same 100 IDs in the same order and identical state/action descriptions.",
        "",
        "## Agreement results",
        "",
        metrics_df.to_markdown(index=False),
        "",
        "## Annotator B provenance context",
        "",
        ANNOTATOR_B_INDEPENDENCE_NOTE,
        "",
        "Annotator B's exact agreement with the automatic labels is reported as an observed result. The owner's confirmation of independent and fair annotation is recorded separately; it is not inferred from kappa or exact agreement.",
        "",
        "## Interpretation and limitations",
        "",
        "- Cohen's kappa measures agreement beyond chance; it does not establish annotator independence.",
        "- The 100 examples are the original annotation batch represented by the uploaded sheets and key.",
        "- Human–automatic comparisons are valid only because the uploaded original key matches all 100 sample IDs.",
        "- The recovered materials do not include the original frozen source JSONL in this upload, so exact source-dataset row provenance beyond the annotator-visible fields and matching key is not independently re-established here.",
        "- Bootstrap intervals are percentile intervals from 5,000 row-resamples; they quantify sampling variability under this resampling procedure.",
        "",
        "## Input file hashes",
        "",
        f"- Annotator A: `{sha256_file(A_ORIGINAL)}`",
        f"- Annotator B: `{sha256_file(B_ORIGINAL)}`",
        f"- Automatic key: `{sha256_file(KEY_ORIGINAL)}`",
    ]
    report = "\n".join(report_lines)
    report_path = REPORTS / "NB6_HUMAN_ANNOTATION_REPORT.md"
    report_path.write_text(report, encoding="utf-8")
    print(report)

# NB6 — Human Annotation Results

- Sample tag: `original_100_annotator_batch`
- Run tag: `recovery_scoring_v1`
- Sample size: 100
- Sample provenance: recovered from the completed annotator sheets and original automatic-label key supplied for this run.
- Both annotator sheets contain the same 100 IDs in the same order and identical state/action descriptions.

## Agreement results

| comparison                                            |   n |   exact_agreement |   cohen_kappa |   ci_95_lower |   ci_95_upper |
|:------------------------------------------------------|----:|------------------:|--------------:|--------------:|--------------:|
| Annotator A vs Annotator B (3-class)                  | 100 |              0.8  |      0.68652  |      0.565959 |      0.801407 |
| Annotator A vs automatic (3-class)                    | 100 |              0.8  |      0.68652  |      0.564267 |      0.799723 |
| Annotator B vs automatic (3-class)                    | 100 |              1    |    

## 9. Validation checklist

In [45]:
if SAMPLE_MODE == "recover_existing":

    auto_lookup = df_key.set_index("id")["label"]
    automatic_labels = (
        ann_a["id"].astype(str).str.strip().map(auto_lookup)
    )

    checks = {
        "annotator_A_has_100_rows": len(ann_a) == 100,

        "annotator_B_has_100_rows": len(ann_b) == 100,

        "annotator_IDs_and_order_match": (
            ann_a["id"].astype(str).str.strip().tolist()
            == ann_b["id"].astype(str).str.strip().tolist()
        ),

        "annotator_visible_content_matches": all(
            ann_a[c].fillna("").astype(str).equals(
                ann_b[c].fillna("").astype(str)
            )
            for c in ["environment", "state_text", "action_text"]
        ),

        "all_automatic_labels_joined": automatic_labels.notna().all(),

        "all_A_labels_valid": set(ann_a["label"]).issubset(ALLOWED_LABELS),

        "all_B_labels_valid": set(ann_b["label"]).issubset(ALLOWED_LABELS),

        "all_automatic_labels_valid": (
            set(automatic_labels.dropna()).issubset(ALLOWED_LABELS)
        ),

        "original_uploads_preserved": all(
            p.exists() for p in [A_ORIGINAL, B_ORIGINAL, KEY_ORIGINAL]
        ),

        "report_created": (
            REPORTS / "NB6_HUMAN_ANNOTATION_REPORT.md"
        ).exists()
    }

    # Convert NumPy/Pandas boolean values to native Python booleans
    checks = {key: bool(value) for key, value in checks.items()}

    print(json.dumps(checks, indent=2))

    if not all(checks.values()):
        raise AssertionError("NB6 validation failed.")

    validation_path = RESULTS / "NB6_validation_checklist.json"

    validation_path.write_text(
        json.dumps(checks, indent=2),
        encoding="utf-8"
    )

    print("\nAll NB6 validation checks passed.")
    print("Saved:", validation_path)

{
  "annotator_A_has_100_rows": true,
  "annotator_B_has_100_rows": true,
  "annotator_IDs_and_order_match": true,
  "annotator_visible_content_matches": true,
  "all_automatic_labels_joined": true,
  "all_A_labels_valid": true,
  "all_B_labels_valid": true,
  "all_automatic_labels_valid": true,
  "original_uploads_preserved": true,
  "report_created": true
}

All NB6 validation checks passed.
Saved: /content/irrev/results/human_annotation/recovery_scoring_v1/NB6_validation_checklist.json


## 10. Timestamped dual Drive backup

In [46]:
if DRIVE_AVAILABLE and SAMPLE_MODE == "recover_existing":

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    snapshot_name = f"NB6_{SAMPLE_TAG}_{RUN_TAG}_{timestamp}"

    # =========================================================
    # REQUIRED BACKUP FILES — ACTUAL RECOVERED PATHS
    # =========================================================

    backup_items = [
        (A_ORIGINAL, "PRIVATE_DO_NOT_SHARE/annotator_A_completed_original.csv"),
        (B_ORIGINAL, "PRIVATE_DO_NOT_SHARE/annotator_B_completed_original.xlsx"),
        (KEY_ORIGINAL, "PRIVATE_DO_NOT_SHARE/automatic_label_key_original.json"),

        (RESULTS / "NB6_agreement_metrics.json", "NB6_agreement_metrics.json"),
        (RESULTS / "NB6_agreement_metrics.csv", "NB6_agreement_metrics.csv"),
        (RESULTS / "NB6_validation_checklist.json", "NB6_validation_checklist.json"),

        (REPORTS / "NB6_HUMAN_ANNOTATION_REPORT.md", "NB6_HUMAN_ANNOTATION_REPORT.md"),
    ]

    # Optional artifacts: include them if they exist
    optional_items = [
        (
            RESULTS / "NB6_row_level_comparison_private.csv",
            "PRIVATE_DO_NOT_SHARE/NB6_row_level_comparison_private.csv"
        ),
        (
            SAMPLE_DIR / "sample_provenance_audit.json",
            "sample_provenance_audit.json"
        ),
    ]

    # =========================================================
    # CHECK ALL REQUIRED FILES BEFORE BACKUP
    # =========================================================

    missing = [
        str(src) for src, _ in backup_items
        if not Path(src).exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Required NB6 backup files are missing:\n"
            + "\n".join(missing)
        )

    backup_items.extend([
        item for item in optional_items
        if Path(item[0]).exists()
    ])

    # =========================================================
    # BACK UP TO BOTH DRIVE ROOTS
    # =========================================================

    backup_results = []

    for root in [DRIVE_ROOT, BACKUP_ROOT]:

        target = Path(root) / snapshot_name

        target.mkdir(parents=True, exist_ok=False)

        verified = []

        for src, rel in backup_items:

            src = Path(src)
            dst = target / rel

            dst.parent.mkdir(parents=True, exist_ok=True)

            shutil.copy2(src, dst)

            source_hash = sha256_file(src)
            destination_hash = sha256_file(dst)

            if source_hash != destination_hash:
                raise IOError(f"SHA-256 mismatch: {dst}")

            verified.append({
                "file": rel,
                "bytes": dst.stat().st_size,
                "sha256": destination_hash
            })

        manifest = {
            "snapshot_name": snapshot_name,
            "sample_tag": SAMPLE_TAG,
            "run_tag": RUN_TAG,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "verified_file_count": len(verified),
            "files_verified": verified,
            "note": (
                "Original annotator sheets and automatic-label key "
                "are stored under PRIVATE_DO_NOT_SHARE. "
                "Do not publish these files."
            )
        }

        manifest_path = target / "NB6_backup_manifest.json"

        manifest_path.write_text(
            json.dumps(manifest, indent=2),
            encoding="utf-8"
        )

        # Verify manifest exists and is readable
        with open(manifest_path, "r", encoding="utf-8") as f:
            json.load(f)

        backup_results.append({
            "snapshot": str(target),
            "verified_files": len(verified),
            "manifest_created": manifest_path.exists(),
            "failures": []
        })

    # =========================================================
    # FINAL OUTPUT
    # =========================================================

    print("=" * 70)
    print("NB6 — DUAL DRIVE BACKUP")
    print("=" * 70)

    print(json.dumps(backup_results, indent=2))

    if len(backup_results) != 2:
        raise AssertionError("Both Drive backups were not completed.")

    if not all(
        result["verified_files"] == len(backup_items)
        and result["manifest_created"]
        and not result["failures"]
        for result in backup_results
    ):
        raise AssertionError("NB6 backup verification failed.")

    print("\nNB6 DUAL BACKUP COMPLETED AND VERIFIED.")

else:
    print("Drive backup skipped (Drive unavailable or generate_new mode).")

NB6 — DUAL DRIVE BACKUP
[
  {
    "snapshot": "/content/drive/MyDrive/ir/NB6_original_100_annotator_batch_recovery_scoring_v1_20261123T135741Z",
    "verified_files": 8,
    "manifest_created": true,
    "failures": []
  },
  {
    "snapshot": "/content/drive/MyDrive/ir_final/NB6_original_100_annotator_batch_recovery_scoring_v1_20261123T135741Z",
    "verified_files": 8,
    "manifest_created": true,
    "failures": []
  }
]

NB6 DUAL BACKUP COMPLETED AND VERIFIED.


## 11. Completion and next-run instructions

- Keep this `SAMPLE_TAG` for any rerun that scores the same original annotation batch.
- Change `RUN_TAG` for a distinct report/backup run; the timestamped snapshot is also unique.
- To intentionally create another sample, set `SAMPLE_MODE = "generate_new"`, choose a **new** `SAMPLE_TAG`, and set `REGENERATE_SAMPLE = True`.
- Never reuse the original sample tag for a new sample.
- The original completed annotator files and key are retained under `incoming_uploads/<SAMPLE_TAG>/`; private material is also stored under `PRIVATE_DO_NOT_SHARE` in Drive snapshots.
- Do not share the private key, row-level comparison file, or full backup snapshot with annotators.